# Notebook 06: Structured Handoffs (CCA Pattern 6)

**CCA Pattern:** Structured EscalationRecord from a schema-enforced tool call vs. raw conversation dump

This notebook demonstrates the final CCA pattern: when an agent escalates to a human, the handoff payload must be a **structured EscalationRecord** — never a raw conversation dump filled with API-internal `tool_use` blocks.

Scenario: C003 Carol Martinez, \$600 refund request (triggers mandatory escalation: amount > $500).

## Setup

In [ ]:
import json
import sys
from pathlib import Path

# Add project root so notebooks.helpers and customer_service are importable
sys.path.insert(0, str(Path(".").resolve()))

import anthropic
from helpers import compare_results, print_usage

from customer_service.agent import (
    build_callbacks,
    get_system_prompt,
    run_agent_loop,
)
from customer_service.anti_patterns import format_raw_handoff
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine

In [ ]:
def make_services() -> ServiceContainer:
    """Create a fresh ServiceContainer with seed customer data."""
    from customer_service.data.customers import CUSTOMERS

    return ServiceContainer(
        customer_db=CustomerDatabase(CUSTOMERS),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


from dotenv import find_dotenv, load_dotenv

# Load ANTHROPIC_API_KEY from .env (find_dotenv walks up from the notebooks/ dir)
load_dotenv(find_dotenv(), override=False)

client = anthropic.Anthropic()

In [ ]:
# C003: Carol Martinez, $600 refund — canonical scenario for escalation
# amount > $500 triggers mandatory escalation regardless of tier
user_message = (
    "Hi, I need to return my purchase. My customer ID is C003 and my order is O003. "
    "The item was defective and I want a refund for the $600 I paid."
)

print("Customer: C003 (Carol Martinez, Regular tier)")
print("Order: O003 — $600 refund request")
print("Expected: escalation (amount > $500 threshold)")
print(f"\nMessage: {user_message}")

## Anti-Pattern: Raw Conversation Dump

<div style="border-left: 4px solid #dc3545; padding: 12px 16px; background: #fff5f5; margin: 8px 0;">
<strong>What's wrong:</strong> Dumping the full <code>messages</code> list as JSON gives the human agent thousands of characters of API-internal noise — <code>tool_use</code> blocks with tool IDs, <code>tool_result</code> blocks with JSON payloads, assistant reasoning fragments. None of this is useful to a human agent. The signal-to-noise ratio is terrible.
</div>

In [ ]:
# Run the agent loop — the $600 refund will end in escalation
services = make_services()
callbacks = build_callbacks()
print("Running agent loop for C003 $600 refund (will escalate)...")
result = run_agent_loop(
    client,
    services,
    user_message,
    get_system_prompt(),
    callbacks=callbacks,
)
print(f"Stop reason: {result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in result.tool_calls]}")

# How did the escalation happen on THIS run? Either way the record has the same shape.
queued = len(services.escalation_queue.get_escalations())
if result.stop_reason == "escalated":
    path = "forced by the loop with tool_choice (the backstop)"
elif queued:
    path = "Claude called escalate_to_human on its own after check_policy flagged review"
else:
    path = "no escalation queued — re-run this cell"
print(f"Escalation path: {path}")

In [ ]:
# Anti-pattern: dump the full messages list as raw JSON
raw_output = format_raw_handoff(result.messages)
raw_len = len(raw_output)

# Show truncated output to illustrate the noise
PREVIEW = 500
preview = raw_output[:PREVIEW]
remaining = raw_len - PREVIEW

print(f"RAW HANDOFF OUTPUT ({raw_len:,} chars total):")
print(preview)
if remaining > 0:
    print(f"... {remaining:,} more chars")
print()
print(f"NOTE: Human agent receives {raw_len:,} chars of raw JSON")
print("      Includes: tool_use blocks, tool IDs, tool_result payloads, API artifacts")

<div style="border-left: 4px solid #dc3545; padding: 12px 16px; background: #fff5f5; margin: 8px 0;">
<strong>ANTI-PATTERN RESULT:</strong> Human agent receives thousands of chars of raw JSON including API-internal <code>tool_use</code> blocks, tool IDs, and <code>tool_result</code> payloads. The actual escalation reason and customer context are buried in the noise.
</div>

## Correct Pattern: Structured EscalationRecord from a schema-enforced tool

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Why this works:</strong> The handoff is whatever <code>escalate_to_human</code> received as input, and the API validates that input against the tool's schema before the handler runs. So the record in <code>escalation_queue</code> always has the same 8 named fields, no matter how the escalation was triggered.
</div>

Two paths lead to the same record, and the cell above prints which one this run took:

1. **Voluntary** (most runs of this scenario): `check_policy` returns `requires_review: true` for $600, and Claude calls `escalate_to_human` itself. The structured record still comes from the schema, not from Claude's judgement.
2. **Forced** (the backstop): if Claude tries `process_refund` anyway, the callback blocks it and sets `action_required = "escalate_to_human"`; the loop then makes one more call with `tool_choice={"type": "tool", "name": "escalate_to_human"}`. The loop also forces escalation if Claude ends its turn with a review flag set and nothing queued.

The guarantee is the **schema**, enforced programmatically on every path. `tool_choice` guarantees the call happens; it is not what makes the payload structured. See Notebook 01 for the escalation rules themselves.

In [ ]:
# Reuse result from above — same run triggered both anti-pattern and correct pattern
# The escalation_queue already has the structured record
all_escalations = services.escalation_queue.get_escalations()
print(f"Escalation queue entries: {len(all_escalations)}")

if all_escalations:
    escalation = all_escalations[-1]
    escalation_dict = escalation.model_dump()
    structured_output = json.dumps(escalation_dict, indent=2, default=str)
    structured_len = len(structured_output)

    print(f"\nSTRUCTURED HANDOFF (EscalationRecord — {structured_len:,} chars):")
    print(structured_output)
else:
    structured_len = 0
    print("No escalation record found — check that $600 amount triggers the > $500 rule")

In [ ]:
class _UsageWrapper:
    def __init__(self, u):
        self.usage = u


print_usage(_UsageWrapper(result.usage))

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>CORRECT PATTERN RESULT:</strong> Human agent receives a clean structured <code>EscalationRecord</code> — 8 named fields: <code>customer_id</code>, <code>customer_tier</code>, <code>issue_type</code>, <code>disputed_amount</code>, <code>escalation_reason</code>, <code>recommended_action</code>, <code>conversation_summary</code>, <code>turns_elapsed</code>. Zero API noise. The field names come from <code>EscalationRecord</code> in <code>models/customer.py</code>, which is also the source of the tool's input schema.
</div>

## Compare: Raw Dump vs Structured Handoff

In [ ]:
if all_escalations and structured_len > 0:
    ratio = raw_len / structured_len
    # Both noise checks are computed, not asserted. Tool IDs start with "toolu_".
    # Metrics are phrased positively so the Delta column reads FIXED, not REGRESSED.
    compare_results(
        {
            "payload_chars": raw_len,
            "free_of_tool_use_blocks": "tool_use" not in raw_output,
            "free_of_tool_ids": "toolu_" not in raw_output,
        },
        {
            "payload_chars": structured_len,
            "free_of_tool_use_blocks": "tool_use" not in structured_output,
            "free_of_tool_ids": "toolu_" not in structured_output,
        },
    )
    print(f"\nRaw dump is {ratio:.1f}x larger than the structured record.")
    print(f"Raw dump: {len(result.messages)} messages of mixed content blocks.")
    print(f"Structured record: {len(escalation_dict)} named fields, all at the top level.")
else:
    print("Comparison skipped — no escalation record found. Re-run the agent loop cell above.")

> **CCA Exam Tip:** Structured JSON handoffs, not raw conversation dumps. The structure is enforced by the tool's input schema; `tool_choice` is the backstop that guarantees the call happens.
>
> - Raw conversation dumps include API-internal `tool_use` blocks, tool IDs, and `tool_result` payloads — noise that human agents cannot use
> - `escalate_to_human` has a fixed input schema, so every record has the same fields whether Claude escalated on its own or was forced with `tool_choice={"type": "tool", "name": "escalate_to_human"}`
> - The `EscalationRecord` has 8 named fields: customer_id, customer_tier, issue_type, disputed_amount, escalation_reason, recommended_action, conversation_summary, turns_elapsed
> - **Exam signal → Answer:** "Human agents can't find the key info" → Raw handoff anti-pattern → Use a structured EscalationRecord produced by a schema-enforced tool call

## Extension: Custom PostToolUse Callback (TODO)

In [ ]:
# TODO: Write a callback that flags interactions with sentiment keywords
# HINTS:
#   1. Add sentiment keywords: ["frustrated", "terrible", "unacceptable"]
#   2. Check context["user_message"] for keywords
#   3. Set context["sentiment_flag"] = True if found
#   4. Return CallbackResult(action="allow") — flag only, don't block
# EXPECTED: Angry customer message sets sentiment_flag in context
def student_sentiment_callback(*args, **kwargs):
    from customer_service.agent.callbacks import CallbackResult

    return CallbackResult(action="allow")  # placeholder — always allows


# Guard: notebook uses default callbacks if TODO not implemented
try:
    # Student replaces the function body above, then uncomment:
    # custom_callbacks = build_callbacks()
    # custom_callbacks["log_interaction"] = student_sentiment_callback
    raise NotImplementedError("TODO: implement sentiment callback")
except NotImplementedError:
    print("TODO not yet implemented - using default callbacks. Core scenario still runs above.")